# Two-Stage Recommendation (Retrieval + Re-ranking)

This notebook completes the architecture. Retrieval identifies a small set of relevant candidates; **re-ranking** optimizes their final order using richer user-item interaction features. The model is LightGBM LambdaMART, but the real subject of this notebook is the two-stage pipeline, not the algorithm.

```
Multiple Retrieval Models  ->  Candidate Fusion  ->  Feature Pipeline  ->  Re-ranking  ->  Top-10
```

Stage in the project: `Classical → Semantic → Neural Retrieval → `**`Learning-to-Rank`**.

## Двухстадийные рекомендации (Retrieval + Re-ranking)

Этот ноутбук завершает архитектуру. Retrieval находит небольшой набор релевантных кандидатов; **re-ranking** оптимизирует их финальный порядок по более богатым признакам пользователь–игра. Модель — LightGBM LambdaMART, но настоящая тема ноутбука — двухстадийный пайплайн, а не алгоритм.

```
Несколько retrieval-моделей  ->  Candidate Fusion  ->  Feature Pipeline  ->  Re-ranking  ->  Top-10
```

Этап проекта: `Classical → Semantic → Neural Retrieval → `**`Learning-to-Rank`**.

In [ ]:
import os
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", message=".*Disabling GPU support.*")
warnings.filterwarnings("ignore", message=".*OpenBLAS.*")
warnings.filterwarnings("ignore", message=".*MKL.*")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT))

from src.baselines import (PopularityRecommender, ALSRecommender,
                           SemanticEmbeddingRecommender, TwoTowerRecommender)
from src.evaluation import evaluate, leave_last_out
from src.ranking import (build_context, build_training_dataset, train_reranker,
                         fuse_candidates, RerankRecommender, FEATURE_NAMES)

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# Pipeline parameters. Defaults are the heavy overnight settings;
# reduce RANK_TRAIN_USERS / K_PER for a quick run.
# Параметры пайплайна. Дефолты — тяжёлые (на ночь);
# уменьши RANK_TRAIN_USERS / K_PER для быстрого прогона.
K_PER = 50               # candidates per retriever (union across sources)
RANK_TRAIN_USERS = None  # None = all users; e.g. 40000 for a quick run
TT_EPOCHS = 10
print("params:", dict(K_PER=K_PER, RANK_TRAIN_USERS=RANK_TRAIN_USERS, TT_EPOCHS=TT_EPOCHS))

## Load Data

Same frozen split, game metadata and semantic game embeddings as the rest of the project.

## Загрузка данных

Тот же зафиксированный split, метаданные игр и семантические эмбеддинги, что и во всём проекте.

In [ ]:
train = pd.read_parquet(DATA / "processed" / "train.parquet")
test = pd.read_parquet(DATA / "processed" / "test.parquet")
games = pd.read_parquet(DATA / "processed" / "games.parquet")
emb_meta = pd.read_parquet(DATA / "processed" / "game_embeddings_metadata.parquet").sort_values("embedding_idx")
game_emb = np.load(DATA / "processed" / "game_embeddings.npy")
game_ids = emb_meta["game_id"].to_numpy()
name_by_id = games.set_index("game_id")["Name"].to_dict()

def fit_retrievers(tr):
    # Fit the four retrieval models that feed the candidate fusion stage
    # Обучаем четыре retrieval-модели, питающие этап candidate fusion
    return {
        "als": ALSRecommender(factors=128, iterations=25, regularization=0.05).fit(tr),
        "two_tower": TwoTowerRecommender(games, game_emb, game_ids, epochs=TT_EPOCHS, seed=42).fit(tr),
        "semantic": SemanticEmbeddingRecommender(game_emb, game_ids).fit(tr),
        "popularity": PopularityRecommender().fit(tr),
    }

print("train:", train.shape, "| test:", test.shape)

## Candidate Fusion

No single retriever is best at everything: popularity captures global trends, ALS collaborative patterns, semantic embeddings gameplay similarity, and the Two-Tower learned user-item structure. Instead of choosing one, we **fuse** their candidates: the union of each retriever's top-K per user. Ranking never sees the whole catalog — only this compact candidate set.

The fused set defines the **ceiling**: the final Recall@10 cannot exceed the candidate recall (whether the held-out game is in the fused set at all).

## Candidate Fusion

Ни один ретривер не лучший во всём: популярность ловит глобальные тренды, ALS — коллаборативные паттерны, семантические эмбеддинги — сходство геймплея, Two-Tower — выученную структуру пользователь–игра. Вместо выбора одного мы **объединяем** их кандидатов: union top-K каждого ретривера на юзера. Ranking никогда не видит весь каталог — только этот компактный набор.

Объединённый набор задаёт **потолок**: финальный Recall@10 не может превысить recall кандидатов (попала ли отложенная игра в набор вообще).

In [ ]:
# Fit retrievers on the full train set (used for fusion and final evaluation)
# Обучаем ретриверы на полном train (для fusion и финальной оценки)
retr_full = fit_retrievers(train)

# Candidate recall ceiling on the test set
# Потолок recall кандидатов на test
test_users = test["user_id"].unique()
truth = test.groupby("user_id")["game_id"].first().to_dict()
fused_test = fuse_candidates(retr_full, test_users, k_per=K_PER)
ceiling = np.mean([truth[u] in set(fused_test[u]) for u in test_users])
avg_cands = np.mean([len(fused_test[u]) for u in test_users])
print(f"avg candidates per user: {avg_cands:.1f}")
print(f"candidate recall ceiling (held-out game in fused set): {ceiling:.4f}")

## Leakage-Free Ranking Dataset

The ranking features include retrieval scores (ALS, semantic, Two-Tower). If we trained the re-ranker on items the retrievers had already seen, those scores would be overly optimistic for the positive examples — a distribution shift between ranking training and final inference.

To avoid this, retrieval models used for ranking-feature generation are trained on a reduced interaction set (`train_fit`). Candidate scores for the held-out ranking targets are therefore produced under the same conditions as during final evaluation (out-of-sample retrieval features).

```
Official Train
        |
        +--------------------+
        |                    |
        v                    v
   train_fit          rank_train_target
        |                    |
        v                    |
  Retrieval Models           |
        |                    |
        +-------> Features <--+
                     |
                     v
                 LambdaMART
```

## Ranking-датасет без утечки

Признаки ранжирования включают retrieval-скоры (ALS, semantic, Two-Tower). Если обучать re-ranker на айтемах, которые ретриверы уже видели, эти скоры были бы завышены для позитивов — сдвиг распределения между обучением ранкера и финальным инференсом.

Чтобы этого избежать, retrieval-модели для генерации ранжирующих признаков обучаются на сокращённом наборе (`train_fit`). Скоры кандидатов для отложенных таргетов получаются в тех же условиях, что и при финальной оценке (out-of-sample retrieval features).

In [ ]:
# Split train -> train_fit + rank_train_target (one held-out interaction per user)
# Делим train -> train_fit + rank_train_target (одно отложенное взаимодействие на юзера)
train_fit, rank_holdout = leave_last_out(train)
print("train_fit:", train_fit.shape, "| rank targets:", rank_holdout.shape)

# Retrievers trained WITHOUT the ranking targets -> leakage-free features
# Ретриверы обучены БЕЗ ранжирующих таргетов -> признаки без утечки
retr_fit = fit_retrievers(train_fit)
ctx_fit = build_context(train_fit, games, game_ids, game_emb, retr_fit["als"], retr_fit["two_tower"])

X, y, groups = build_training_dataset(rank_holdout, retr_fit, ctx_fit,
                                      k_per=K_PER, n_users=RANK_TRAIN_USERS, seed=42)
print("ranking dataset:", X.shape, "| positives:", int(y.sum()), "| query groups:", len(groups))

## Feature Pipeline

Each (user, candidate) pair is described by features grouped by source. They are defined as small functions in `src/ranking/features.py` and registered in `FEATURE_BUILDERS`, so adding or removing a feature is a one-line change.

- **Retrieval features**: `als_score`, `semantic_score`, `two_tower_score`, `popularity_rank`
- **Content features**: `genre_overlap`, `tag_overlap`, `price_difference`
- **Game features**: `review_score`, `game_popularity`, `price`
- **User features**: `avg_playtime`, `positive_ratio`, `n_games`

## Feature Pipeline

Каждая пара (user, candidate) описывается признаками, сгруппированными по источнику. Они заданы маленькими функциями в `src/ranking/features.py` и зарегистрированы в `FEATURE_BUILDERS`, поэтому добавить/убрать признак — это одна строка.

- **Retrieval features**: `als_score`, `semantic_score`, `two_tower_score`, `popularity_rank`
- **Content features**: `genre_overlap`, `tag_overlap`, `price_difference`
- **Game features**: `review_score`, `game_popularity`, `price`
- **User features**: `avg_playtime`, `positive_ratio`, `n_games`

In [ ]:
print("features:", FEATURE_NAMES)
X.describe().T[["mean", "std", "min", "max"]].round(3)

## Feature Analysis

A quick look at how the features relate to each other before training — confirming they carry different signals rather than duplicating one another.

## Анализ признаков

Быстрый взгляд на связь признаков до обучения — подтверждаем, что они несут разные сигналы, а не дублируют друг друга.

In [ ]:
corr = X.corr()
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=8)
ax.grid(False)
fig.colorbar(im, fraction=0.046, pad=0.04)
ax.set_title("Feature correlation")
plt.tight_layout()
plt.show()

## LambdaMART

**Why Learning-to-Rank?** We do not need to predict a score per item in isolation; we need the *order* of the candidate list to be good. LambdaMART optimizes a ranking objective (NDCG) directly.

**Why query groups?** Each user is a query: the model ranks that user's candidates against each other, not across users. The group sizes tell LightGBM which rows belong to the same user.

## LambdaMART

**Зачем Learning-to-Rank?** Нам не нужно предсказывать скор каждого айтема по отдельности — нужен хороший *порядок* списка кандидатов. LambdaMART оптимизирует ранжирующую метрику (NDCG) напрямую.

**Зачем query-группы?** Каждый юзер — это query: модель ранжирует кандидатов внутри юзера, а не между юзерами. Размеры групп говорят LightGBM, какие строки относятся к одному юзеру.

## Training

## Обучение

In [ ]:
# Train the LambdaMART re-ranker on the leakage-free dataset
# Обучаем re-ranker LambdaMART на датасете без утечки
rr_model = train_reranker(X, y, groups)

# Build the context on the full train set and assemble the two-stage recommender
# Строим контекст на полном train и собираем двухстадийный рекомендер
ctx_full = build_context(train, games, game_ids, game_emb, retr_full["als"], retr_full["two_tower"])
reranker = RerankRecommender(retr_full, ctx_full, rr_model, k_per=K_PER)
print("reranker ready")

## Feature Importance

## Важность признаков

In [ ]:
imp = pd.Series(rr_model.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
imp.plot.barh(ax=ax, color="#4C72B0")
ax.set_xlabel("importance (gain)")
ax.set_title("LambdaMART feature importance")
plt.tight_layout()
plt.show()
imp.sort_values(ascending=False)

The re-ranker combines signals from collaborative retrieval, semantic similarity and structured metadata rather than relying on a single retrieval strategy.

Re-ranker объединяет сигналы коллаборативного retrieval, семантического сходства и структурных метаданных, а не опирается на одну retrieval-стратегию.

## Final Benchmark

The table the whole project was built for: every stage on the same leave-last-out protocol, ending with the two-stage pipeline.

## Финальный бенчмарк

Таблица, ради которой строился весь проект: каждый этап на одном leave-last-out протоколе, финал — двухстадийный пайплайн.

In [ ]:
K = 10
rows = {
    ("Classical Retrieval", "ALS"): retr_full["als"],
    ("Semantic Retrieval", "Semantic Embedding"): retr_full["semantic"],
    ("Neural Retrieval", "Two-Tower"): retr_full["two_tower"],
    ("Two-Stage Recommendation", "Retrieval + Re-ranking"): reranker,
}
pop_res = evaluate(retr_full["popularity"], train, test, k=K)

records = [("Baseline", "Popularity", *pop_res.values())]
for (stage, model_name), rec in rows.items():
    r = evaluate(rec, train, test, k=K)
    records.append((stage, model_name, r["Recall@10"], r["MAP@10"], r["NDCG@10"]))

final = pd.DataFrame(records, columns=["Stage", "Best Model", "Recall@10", "MAP@10", "NDCG@10"]).round(4)
print(final.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(final["Best Model"], final["Recall@10"],
       color=["#BBBBBB", "#4C72B0", "#8172B3", "#CCB974", "#C44E52"])
ax.set_ylabel("Recall@10")
ax.set_title("Final benchmark (Recall@10)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## Architecture Summary

```
                 User
                  |
                  v
   +-----------------------------+
   |   Multi-Source Retrieval    |
   |  Popularity / ALS /         |
   |  Semantic / Two-Tower       |
   +-----------------------------+
                  |
                  v
          Candidate Fusion
                  |
                  v
          Feature Pipeline
                  |
                  v
        LambdaMART Re-ranking
                  |
                  v
            Top-10 Recommendations
```

Final production-style recommendation pipeline: cheap retrieval narrows the catalog, an expensive ranker orders only the survivors.

## Сводка архитектуры

Финальный production-style пайплайн: дешёвый retrieval сужает каталог, дорогой ранкер упорядочивает только прошедших отбор.

## Discussion

_Filled in after the overnight run, based on the final benchmark numbers._

## Обсуждение

_Заполняется после ночного прогона, по числам финального бенчмарка._

## Project Summary

**What this project is.** An end-to-end, production-style game recommender built as an evolution of representations: classical sparse features, semantic review embeddings, learned neural embeddings, and finally a two-stage retrieval + re-ranking pipeline — all evaluated under one fixed leave-last-out protocol.

**What worked.**
- A single evaluation protocol fixed up front, reused by every model, making comparisons fair.
- BM25-weighted ALS is a very strong retrieval baseline on this data.
- Review-text embeddings produce a semantically meaningful game space.
- Multi-source candidate fusion plus a leakage-free ranking dataset (out-of-sample retrieval features) turns the separate signals into a single ranking layer.

**Limitations.**
- Small, sparse, temporally split dataset (~6 interactions per user, ~2.8k games).
- No user identity embeddings in the Two-Tower model.
- Static 2021 data snapshot; no online/feedback signals.

**Future work.**
- User ID embeddings and larger interaction data for the neural retriever.
- Session/temporal features in the ranker.
- Serving: FastAPI + Docker around the two-stage pipeline.

## Итоги проекта

**Что это за проект.** End-to-end рекомендательная система production-уровня, построенная как эволюция представлений: классические разреженные признаки, семантические эмбеддинги отзывов, обученные нейронные эмбеддинги и, наконец, двухстадийный пайплайн retrieval + re-ranking — всё под одним зафиксированным leave-last-out протоколом.

**Что сработало.**
- Единый протокол оценки, зафиксированный заранее и переиспользованный всеми моделями.
- BM25-взвешенный ALS — очень сильный retrieval-baseline на этих данных.
- Эмбеддинги текста отзывов дают осмысленное семантическое пространство игр.
- Multi-source candidate fusion + ranking-датасет без утечки (out-of-sample retrieval features) объединяют разные сигналы в один слой ранжирования.

**Ограничения.**
- Маленький, разреженный, временной датасет (~6 взаимодействий на юзера, ~2.8k игр).
- Нет ID-эмбеддингов пользователей в Two-Tower.
- Статический срез данных 2021; нет онлайн/feedback сигналов.

**Дальнейшая работа.**
- ID-эмбеддинги пользователей и больше данных для нейронного ретривера.
- Сессионные/временные признаки в ранкере.
- Сервинг: FastAPI + Docker вокруг двухстадийного пайплайна.